<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/Lab2aml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup and Preparation

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda:0


## Data Augmentation and Dataloaders

In [11]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

if not os.path.exists('hymenoptera_data'):
    !wget https://download.pytorch.org/tutorial/hymenoptera_data.zip
    !unzip hymenoptera_data.zip
    !rm hymenoptera_data.zip

data_dir = 'hymenoptera_data'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=2) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes




In [12]:
def train_model(model, criterion, optimizer, scheduler=None, num_epochs=3):
  for epoch in range(num_epochs):
    print(f'Epoch {epoch}/{num_epochs - 1}')
    print('-' * 10)

    for phase in ['train', 'val']:
      if phase == 'train':
        model.train()
      else:
        model.eval()

      running_loss = 0.0
      running_corrects = 0

      for inputs, labels in dataloaders[phase]:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()

        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          _, preds = torch.max(outputs, 1)
          loss = criterion(outputs, labels)

          if phase == 'train':
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
      if phase == 'train' and scheduler is not None:
        scheduler.step()

      epoch_loss = running_loss / dataset_sizes[phase]
      epoch_acc = running_corrects.double() / dataset_sizes[phase]

      print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

  return model

## Design: CNN vs MLP

MLP Model

In [13]:
class MLPModel(nn.Module):
  def __init__(self):
    super(MLPModel, self).__init__()
    self.flatten = nn.Flatten()
    self.fc = nn.Sequential(
        nn.Linear(224*224*3, 512),
        nn.ReLU(),
        nn.Linear(512, len(class_names))
    )

  def forward(self, x):
    x = self.flatten(x)
    return self.fc(x)

Custom CNN Model

In [14]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN, self).__init__()
    self.features = nn.Sequential (
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )
    self.classifier = nn.Sequential(
        nn.Linear(32*56*56, 128),
        nn.ReLU(),
        nn.Linear(128, len(class_names))
    )

  def forward(self, x):
    x = self.features(x)
    x = x.view(x.size(0), -1)
    return self.classifier(x)


In [15]:
criterion = nn.CrossEntropyLoss()

print("--- Training MLP Model ---")
model_mlp = MLPModel().to(device)
optimizer_mlp = optim.Adam(model_mlp.parameters(), lr=0.001)
train_model(model_mlp, criterion, optimizer_mlp, num_epochs=3)

print("--- Training Simple CNN Model ---")
model_cnn = SimpleCNN().to(device)
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=0.001)
train_model(model_cnn, criterion, optimizer_cnn, num_epochs=3)




--- Training MLP Model ---
Epoch 0/2
----------
train Loss: 74.9035 Acc: 0.5287
val Loss: 47.5369 Acc: 0.5556
Epoch 1/2
----------
train Loss: 21.0124 Acc: 0.5574
val Loss: 24.9338 Acc: 0.5425
Epoch 2/2
----------
train Loss: 5.9115 Acc: 0.4918
val Loss: 23.0219 Acc: 0.5033
--- Training Simple CNN Model ---
Epoch 0/2
----------
train Loss: 1.1837 Acc: 0.5328
val Loss: 0.6648 Acc: 0.6144
Epoch 1/2
----------
train Loss: 0.6607 Acc: 0.6107
val Loss: 0.6661 Acc: 0.6536
Epoch 2/2
----------
train Loss: 0.6320 Acc: 0.6844
val Loss: 0.7643 Acc: 0.6013


SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=100352, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)

## Fine-tuning Pretrained Models

Method 1: Freezing Weights (ResNet18)

In [16]:
print("\n--- Training ResNet18 ---")
model_resnet = models.resnet18(pretrained=True)
for param in model_resnet.parameters():
  param.requires_grad = False

num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_ftrs, len(class_names))
model_resnet = model_resnet.to(device)

optimizer_res = optim.SGD(model_resnet.fc.parameters(), lr=0.001, momentum=0.9)
train_model(model_resnet, criterion, optimizer_res, num_epochs=3)



--- Training ResNet18 ---


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 0/2
----------
train Loss: 0.6673 Acc: 0.6270
val Loss: 0.2675 Acc: 0.9281
Epoch 1/2
----------
train Loss: 0.5123 Acc: 0.7172
val Loss: 0.4881 Acc: 0.7908
Epoch 2/2
----------
train Loss: 0.4424 Acc: 0.8115
val Loss: 0.2865 Acc: 0.8889


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Method 2: Reconstructing Layers (MobileNetV2)

In [17]:
print("\n--- Training MobileNet V2 ")
model_mobilenet = models.mobilenet_v2(pretrained=True)
num_ftrs = model_mobilenet.classifier[1].in_features
model_mobilenet.classifier[1] = nn.Linear(num_ftrs, len(class_names))
model_mobilenet = model_mobilenet.to(device)

optimizer_mob = optim.SGD(model_mobilenet.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_mob, step_size=7, gamma=0.1)
train_model(model_mobilenet, criterion, optimizer_mob, scheduler=exp_lr_scheduler, num_epochs=3)


--- Training MobileNet V2 
Epoch 0/2
----------


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


train Loss: 0.7152 Acc: 0.6311
val Loss: 0.3581 Acc: 0.8693
Epoch 1/2
----------
train Loss: 0.5486 Acc: 0.7746
val Loss: 0.3689 Acc: 0.8431
Epoch 2/2
----------
train Loss: 0.6522 Acc: 0.7254
val Loss: 0.4549 Acc: 0.8431


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

## Training Loop and Results

In [18]:
criterion = nn.CrossEntropyLoss()

print("--- Training ResNet18 (Method 1: Frozen Weights) ---")
optimizer_res = optim.SGD(model_resnet.fc.parameters(), lr=0.001, momentum=0.9)
model_resnet = train_model(model_resnet, criterion, optimizer_res)

print("--- Training MobileNet V2 (Method 2: Full Fine-tuning) ---")
optimizer_mob = optim.SGD(model_mobilenet.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_mob, step_size=7, gamma=0.1)
model_mobilenet = train_model(model_mobilenet, criterion, optimizer_mob)

--- Training ResNet18 (Method 1: Frozen Weights) ---
Epoch 0/2
----------
train Loss: 0.3660 Acc: 0.8443
val Loss: 0.2974 Acc: 0.8954
Epoch 1/2
----------
train Loss: 0.3357 Acc: 0.8525
val Loss: 0.2472 Acc: 0.9346
Epoch 2/2
----------
train Loss: 0.3909 Acc: 0.8238
val Loss: 0.2693 Acc: 0.9216
--- Training MobileNet V2 (Method 2: Full Fine-tuning) ---
Epoch 0/2
----------
train Loss: 0.6006 Acc: 0.7213
val Loss: 0.3116 Acc: 0.8954
Epoch 1/2
----------
train Loss: 0.6288 Acc: 0.7377
val Loss: 0.4124 Acc: 0.8235
Epoch 2/2
----------
train Loss: 0.6910 Acc: 0.7213
val Loss: 0.4869 Acc: 0.8105
